this script will:

- upload four geotiffs from google earth engine to google drive
- (then you manually download them locally)
- merges the four geotiffs into one
- convert from 32 bit to 8 bit and EPSG:4326 to EPSG:3857
- generate the tiles from the geotiff and put them in tiles/{x}/{y}/{z}


In [8]:
import ee

ee.Initialize(project="gsapp-map")

YEAR = 2023
SCALE = 1000  # meters per pixel

img = (
    ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
    .filterDate(f"{YEAR}-01-01", f"{YEAR}-12-31")
    .mosaic()
    .select(["A01", "A02", "A03"])
)

regions = [
    ee.Geometry.Rectangle([-180, -85, -90, 85]),
    ee.Geometry.Rectangle([-90, -85, 0, 85]),
    ee.Geometry.Rectangle([0, -85, 90, 85]),
    ee.Geometry.Rectangle([90, -85, 180, 85]),
]

for i, reg in enumerate(regions):
    task = ee.batch.Export.image.toDrive(
        image=img,
        description=f"alphaearth_{YEAR}_{SCALE}_rgb_part{i}",
        folder="AlphaEarthTiles",
        fileNamePrefix=f"alphaearth_{YEAR}_{SCALE}_rgb_part{i}",
        region=reg,
        scale=SCALE,  # m per pixel
        crs="EPSG:4326",
        maxPixels=1e13,
    )
    task.start()
    print(f"Started export for region {i}")

print("Track progress at https://code.earthengine.google.com/tasks.")

Started export for region 0
Started export for region 1
Started export for region 2
Started export for region 3
Track progress at https://code.earthengine.google.com/tasks.


In [1]:
import glob
import os
from osgeo import gdal, gdalconst
import numpy as np
import subprocess


YEAR = 2023
SCALE = 1000

# --- Step 0: Set file paths ---
input_files = sorted(glob.glob(f"tifs/alphaearth_{YEAR}_{SCALE}_rgb_part*.tif"))
merged_file = f"tifs/alphaearth_{YEAR}_{SCALE}_merged.tif"
merged_8bit_file = f"tifs/alphaearth_{YEAR}_{SCALE}_merged_8bit.tif"
warped_file = f"tifs/alphaearth_{YEAR}_{SCALE}_merged_8bit_warped.tif"
tiles_dir = f"tiles/{YEAR}_{SCALE}"

# --- Step 1: Merge all parts into one ---
print("Merging TIFF parts...")
vrt = gdal.BuildVRT("/vsimem/merged.vrt", input_files)
gdal.Translate(merged_file, vrt)
print(f"Merged file created: {merged_file}")

# --- Step 1b: sanity check ---
ds = gdal.Open(merged_file)
print("Merged file info:")
print(f"Raster size: {ds.RasterXSize} x {ds.RasterYSize}")
print(f"Bands: {ds.RasterCount}")
print(f"Projection: {ds.GetProjection()}")
ds = None  # close dataset

# --- Step 2: Convert to 8-bit RGB ---
print("Converting to 8-bit RGB...")
# Read merged dataset
ds = gdal.Open(merged_file)
bands = [ds.GetRasterBand(i + 1).ReadAsArray() for i in range(3)]
# Scale each band from [-0.3,0.3] -> [0,255]

scaled_bands = []
for b in bands:
    b_scaled = ((b + 0.3) / 0.6 * 255).clip(0, 255).astype(np.uint8)
    scaled_bands.append(b_scaled)
# Create output 8-bit RGB
driver = gdal.GetDriverByName("GTiff")
out_ds = driver.Create(
    merged_8bit_file,
    ds.RasterXSize,
    ds.RasterYSize,
    3,
    gdal.GDT_Byte,
    options=["PHOTOMETRIC=RGB"],
)
for i, b in enumerate(scaled_bands):
    out_ds.GetRasterBand(i + 1).WriteArray(b)
out_ds.SetGeoTransform(ds.GetGeoTransform())
out_ds.SetProjection(ds.GetProjection())
out_ds.FlushCache()
out_ds = None
ds = None
print(f"8-bit RGB file created: {merged_8bit_file}")

# --- Step 3: Warp to Web Mercator (EPSG:3857) ---
print("Warping to Web Mercator (EPSG:3857)...")
warp_options = gdal.WarpOptions(
    dstSRS="EPSG:3857",
    outputBounds=[
        -20037508.3427892,
        -20037508.3427892,
        20037508.3427892,
        20037508.3427892,
    ],
    xRes=5000,
    yRes=5000,
    resampleAlg=gdalconst.GRA_Bilinear,
)
gdal.Warp(warped_file, merged_8bit_file, options=warp_options)
print(f"Warped file created: {warped_file}")

# --- Step 3b: sanity check ---
ds = gdal.Open(warped_file)
print("Warped file info:")
print(f"Raster size: {ds.RasterXSize} x {ds.RasterYSize}")
print(f"Bands: {ds.RasterCount}")
print(f"Projection: {ds.GetProjection()}")
ds = None

# --- Step 4: Generate XYZ tiles for MapLibre ---
print("Generating XYZ tiles for MapLibre...")
os.makedirs(tiles_dir, exist_ok=True)

subprocess.run(
    ["gdal2tiles.py", "--xyz", "-z", "0-8", warped_file, tiles_dir], check=True
)

print(f"Tiles generated in folder: {tiles_dir}")

Merging TIFF parts...


/Users/elliemadsen/miniforge3/envs/cdp/lib/python3.12/site-packages/osgeo/gdal.py:330: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Merged file created: tifs/alphaearth_2023_1000_merged.tif
Merged file info:
Raster size: 40076 x 19250
Bands: 3
Projection: GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Latitude",NORTH],AXIS["Longitude",EAST],AUTHORITY["EPSG","4326"]]
Converting to 8-bit RGB...


/var/folders/q5/x4rs4g890zn1_dtyc2sff4r40000gn/T/ipykernel_4546/1168974356.py:41: RuntimeWarning: invalid value encountered in cast
  b_scaled = ((b + 0.3) / 0.6 * 255).clip(0, 255).astype(np.uint8)


8-bit RGB file created: tifs/alphaearth_2023_1000_merged_8bit.tif
Warping to Web Mercator (EPSG:3857)...
Warped file created: tifs/alphaearth_2023_1000_merged_8bit_warped.tif
Warped file info:
Raster size: 8015 x 8015
Bands: 3
Projection: PROJCS["WGS 84 / Pseudo-Mercator",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Mercator_1SP"],PARAMETER["central_meridian",0],PARAMETER["scale_factor",1],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],EXTENSION["PROJ4","+proj=merc +a=6378137 +b=6378137 +lat_ts=0 +lon_0=0 +x_0=0 +y_0=0 +k=1 +units=m +nadgrids=@null +wktext +no_defs"],AUTHORITY["EPSG","3857"]]
Generating XYZ tiles for MapLibre...


Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done in 00:13:09.
0

Generating Overview Tiles:


...10...20...30...40...50...60...70...80...90...100 - done in 00:03:36.
Tiles generated in folder: tiles/2023_1000
